In [1]:
import os
import json
import time
from openai import OpenAI

API_KEY = "sk-proj-xxxxxxxxxxxxxxxxx" 
client = OpenAI(api_key=API_KEY)

In [2]:
def generate_gold_label(model, system_prompt, user_content):
    """Helper for generating strict answer keys"""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=0.0 # Strict deterministic output
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"  [!] API Error: {e}")
        return None

In [3]:
# API connection test
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Say hello in one word"
)

print(response.output_text)

Hello!


In [4]:
# SROIE (Test Set - Gold Labels) 
INPUT_FILE = "phase1_sroie_input_test.jsonl"
OUTPUT_FILE = "phase1b_sroie_test_gold.jsonl"
MODEL = "gpt-4.1"

SYSTEM_PROMPT = """You are an expert data extractor.
Task: Extract merchant, date, total, and address from the receipt text.
Output Format: Return ONLY the raw JSON object. Do NOT write '```json' or explanations."""

In [5]:
print(f"Generating Gold Labels for {INPUT_FILE}...")

if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r') as f_in, open(OUTPUT_FILE, 'w') as f_out:
        lines = f_in.readlines()
        print(f"  -> Found {len(lines)} test samples.")
        
        for i, line in enumerate(lines):
            data = json.loads(line)
            
            if data.get('output') == "TO_BE_GENERATED":
                output = generate_gold_label(MODEL, SYSTEM_PROMPT, data['input'])
                if output:
                
                    clean_output = output.replace("```json", "").replace("```", "").strip()
                    data['output'] = clean_output
            
            f_out.write(json.dumps(data) + "\n")
            
            if (i+1) % 20 == 0: 
                print(f"  [SROIE Test] Processed {i+1}/{len(lines)}")

    print(f"SROIE Test Key saved to {OUTPUT_FILE}")
else:
    print(f"Error: {INPUT_FILE} not found.")

Generating Gold Labels for phase1_sroie_input_test.jsonl...
  -> Found 126 test samples.
  [SROIE Test] Processed 20/126
  [SROIE Test] Processed 40/126
  [SROIE Test] Processed 60/126
  [SROIE Test] Processed 80/126
  [SROIE Test] Processed 100/126
  [SROIE Test] Processed 120/126
SROIE Test Key saved to phase1b_sroie_test_gold.jsonl
